# Pearls AQI Predictor - Exploratory Data Analysis

**City:** Sargodha, Pakistan (32.08°N, 72.67°E)  
**Dataset:** 43,801 hourly observations (July 2021 - July 2026)  
**Features:** 47 engineered columns from AQICN + OpenWeatherMap APIs

---

## Table of Contents
1. Data Loading & Overview
2. Missing Value Analysis
3. Descriptive Statistics
4. Distribution Analysis
5. Correlation Analysis
6. Temporal Pattern Analysis
7. Seasonal Decomposition & Stationarity
8. Weather-Pollutant Relationships
9. Autocorrelation (ACF/PACF)
10. Hypothesis Testing
11. Feature Importance (Mutual Information)
12. Outlier Detection
13. Key Findings & Modeling Implications

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
%matplotlib inline

print('Libraries loaded successfully')

## 1. Data Loading & Overview

In [ ]:
DATA_PATH = Path('../data/backfill_features.csv')
df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

print(f'Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Time Range: {df["timestamp"].min()} to {df["timestamp"].max()}')
print(f'Duration: {(df["timestamp"].max() - df["timestamp"].min()).days:,} days')
print(f'Memory Usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print(f'\nColumn Types:')
print(df.dtypes.value_counts())

In [ ]:
df.head(10)

In [ ]:
print('Columns:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col} ({df[col].dtype})')

## 2. Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_report = missing_report[missing_report['Missing Count'] > 0]

if len(missing_report) == 0:
    print('No missing values found in the dataset!')
    print(f'Total cells: {df.shape[0] * df.shape[1]:,}')
    print('Data completeness: 100%')
else:
    print(missing_report.sort_values('Missing %', ascending=False))

## 3. Descriptive Statistics

In [ ]:
pollutants = ['aqi_value', 'pm25', 'pm10', 'no2', 'so2', 'co', 'o3']
weather = ['temperature_c', 'humidity_pct', 'wind_speed_ms', 'wind_direction_deg', 'pressure_hpa', 'precipitation_mm']

print('=== Pollutant Statistics ===')
desc = df[pollutants].describe(percentiles=[.05, .25, .5, .75, .95]).T
desc['skewness'] = df[pollutants].skew()
desc['kurtosis'] = df[pollutants].kurtosis()
desc

In [ ]:
print('=== Weather Statistics ===')
desc_w = df[weather].describe(percentiles=[.05, .25, .5, .75, .95]).T
desc_w['skewness'] = df[weather].skew()
desc_w['kurtosis'] = df[weather].kurtosis()
desc_w

## 4. Distribution Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Distribution of Key Pollutants - Sargodha, Pakistan', fontsize=14, fontweight='bold')

colors = ['#e74c3c', '#8e44ad', '#2980b9', '#27ae60', '#f39c12', '#1abc9c']
for ax, col, color in zip(axes.flat, pollutants[:6], colors):
    data = df[col].dropna()
    ax.hist(data, bins=50, color=color, alpha=0.7, edgecolor='white', density=True)
    kde_x = np.linspace(data.min(), data.max(), 200)
    kde = stats.gaussian_kde(data)
    ax.plot(kde_x, kde(kde_x), color='black', lw=2)
    ax.set_title(f'{col}\n(mean={data.mean():.1f}, std={data.std():.1f})')
    _, p_val = stats.normaltest(data.sample(min(5000, len(data)), random_state=42))
    ax.text(0.95, 0.95, f'Normal p={p_val:.2e}', transform=ax.transAxes,
            ha='right', va='top', fontsize=8, style='italic')

plt.tight_layout()
plt.show()

In [ ]:
# AQI Category Distribution
bins = [0, 50, 100, 150, 200, 300, 500]
labels = ['Good', 'Moderate', 'Unhealthy-SG', 'Unhealthy', 'Very Unhealthy', 'Hazardous']
df['aqi_category'] = pd.cut(df['aqi_value'], bins=bins, labels=labels)

cat_counts = df['aqi_category'].value_counts()
cat_pct = (cat_counts / len(df) * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 6))
colors_cat = ['#00e400', '#ffff00', '#ff7e00', '#ff0000', '#8f3f97', '#7e0023']
bars = ax.bar(cat_pct.index.astype(str), cat_pct.values, color=colors_cat[:len(cat_pct)])
ax.set_title('AQI Category Distribution (EPA Standards)', fontsize=13, fontweight='bold')
ax.set_ylabel('Percentage (%)')
for bar, pct in zip(bars, cat_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{pct:.1f}%', ha='center')
plt.tight_layout()
plt.show()

print('\nAQI Category Breakdown:')
print(pd.DataFrame({'Count': cat_counts, 'Percentage': cat_pct}))

## 5. Correlation Analysis

In [ ]:
key_cols = ['aqi_value', 'pm25', 'pm10', 'no2', 'so2', 'co', 'o3',
            'temperature_c', 'humidity_pct', 'wind_speed_ms', 'pressure_hpa']
corr = df[key_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix: Pollutants & Weather', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTop Correlations with AQI:')
aqi_corr = corr['aqi_value'].drop('aqi_value').sort_values(key=abs, ascending=False)
print(aqi_corr.round(3))

## 6. Temporal Pattern Analysis

In [ ]:
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['month_num'] = df['timestamp'].dt.month

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Temporal Patterns in AQI', fontsize=14, fontweight='bold')

# Hourly
hourly = df.groupby('hour')['aqi_value'].agg(['mean', 'std']).reset_index()
axes[0,0].plot(hourly['hour'], hourly['mean'], 'o-', color='#e74c3c', lw=2)
axes[0,0].fill_between(hourly['hour'], hourly['mean']-hourly['std'],
                        hourly['mean']+hourly['std'], alpha=0.2, color='#e74c3c')
axes[0,0].set_title('Diurnal Pattern')
axes[0,0].set_xlabel('Hour')
axes[0,0].set_ylabel('AQI')

# Weekly
days = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
weekly = df.groupby('day_of_week')['aqi_value'].mean()
axes[0,1].bar(range(7), weekly.values, color=['#3498db']*5+['#e74c3c']*2)
axes[0,1].set_xticks(range(7))
axes[0,1].set_xticklabels(days)
axes[0,1].set_title('Weekly Pattern')
axes[0,1].set_ylabel('Mean AQI')

# Monthly
months_label = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly = df.groupby('month_num')['aqi_value'].mean()
axes[1,0].bar(range(1,13), monthly.values, color='#27ae60', alpha=0.8)
axes[1,0].set_xticks(range(1,13))
axes[1,0].set_xticklabels(months_label, rotation=45)
axes[1,0].set_title('Seasonal Pattern')
axes[1,0].set_ylabel('Mean AQI')
axes[1,0].axhline(100, color='orange', ls='--', alpha=0.7)

# Yearly
yearly = df.groupby(df['timestamp'].dt.year)['aqi_value'].agg(['mean','std'])
axes[1,1].plot(yearly.index, yearly['mean'], 's-', color='#8e44ad', lw=2)
axes[1,1].fill_between(yearly.index, yearly['mean']-yearly['std'],
                        yearly['mean']+yearly['std'], alpha=0.2, color='#8e44ad')
axes[1,1].set_title('Yearly Trend')
axes[1,1].set_ylabel('Mean AQI')

plt.tight_layout()
plt.show()

In [ ]:
# Hour x Month Heatmap
pivot = df.pivot_table(values='aqi_value', index='hour', columns='month_num', aggfunc='mean')

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd', interpolation='bicubic')
ax.set_xticks(range(12))
ax.set_xticklabels(months_label)
ax.set_yticks(range(24))
ax.set_yticklabels([f'{h:02d}:00' for h in range(24)])
ax.set_xlabel('Month')
ax.set_ylabel('Hour of Day')
ax.set_title('AQI Heatmap: Hour of Day vs Month', fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8, label='Mean AQI')
plt.tight_layout()
plt.show()

## 7. Seasonal Decomposition & Stationarity

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

daily_aqi = df.set_index('timestamp')['aqi_value'].resample('D').mean().dropna()
decomp = seasonal_decompose(daily_aqi, model='additive', period=365)

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
fig.suptitle('Time Series Decomposition (Period=365 days)', fontweight='bold')
axes[0].plot(decomp.observed, color='#2c3e50', lw=0.8)
axes[0].set_ylabel('Observed')
axes[1].plot(decomp.trend, color='#e74c3c', lw=1.5)
axes[1].set_ylabel('Trend')
axes[2].plot(decomp.seasonal, color='#27ae60', lw=0.5)
axes[2].set_ylabel('Seasonal')
axes[3].plot(decomp.resid, color='#7f8c8d', lw=0.5)
axes[3].set_ylabel('Residual')
plt.tight_layout()
plt.show()

# ADF Test
adf = adfuller(daily_aqi, autolag='AIC')
print('\nAugmented Dickey-Fuller Test:')
print(f'  ADF Statistic: {adf[0]:.4f}')
print(f'  p-value: {adf[1]:.6f}')
print(f'  Critical Values:')
for key, val in adf[4].items():
    print(f'    {key}: {val:.4f}')
print(f'\n  Conclusion: Series is {"STATIONARY" if adf[1] < 0.05 else "NON-STATIONARY"}')

## 8. Weather-Pollutant Relationships

In [ ]:
df['season'] = df['month_num'].map({
    12:'Winter', 1:'Winter', 2:'Winter',
    3:'Spring', 4:'Spring', 5:'Spring',
    6:'Summer', 7:'Summer', 8:'Summer',
    9:'Autumn', 10:'Autumn', 11:'Autumn'
})

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Weather Impact on Air Quality', fontweight='bold')

for season, color in zip(['Winter','Spring','Summer','Autumn'],
                         ['#3498db','#27ae60','#e74c3c','#f39c12']):
    subset = df[df['season']==season]
    axes[0,0].scatter(subset['temperature_c'], subset['aqi_value'],
                      alpha=0.05, s=3, color=color, label=season)
axes[0,0].set_xlabel('Temperature (C)')
axes[0,0].set_ylabel('AQI')
axes[0,0].set_title('Temperature vs AQI by Season')
axes[0,0].legend(markerscale=5)

wind_bins = pd.cut(df['wind_speed_ms'], bins=10)
wind_aqi = df.groupby(wind_bins, observed=True)['aqi_value'].mean()
axes[0,1].bar(range(len(wind_aqi)), wind_aqi.values, color='#1abc9c')
axes[0,1].set_xlabel('Wind Speed Bin')
axes[0,1].set_ylabel('Mean AQI')
axes[0,1].set_title('Wind Speed vs AQI')

axes[1,0].scatter(df['humidity_pct'], df['pm25'], alpha=0.05, s=3, color='#8e44ad')
axes[1,0].set_xlabel('Humidity (%)')
axes[1,0].set_ylabel('PM2.5')
axes[1,0].set_title('Humidity vs PM2.5')

axes[1,1].scatter(df['pressure_hpa'], df['aqi_value'], alpha=0.05, s=3, color='#e67e22')
axes[1,1].set_xlabel('Pressure (hPa)')
axes[1,1].set_ylabel('AQI')
axes[1,1].set_title('Pressure vs AQI')

plt.tight_layout()
plt.show()

## 9. Autocorrelation (ACF/PACF)

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
plot_acf(daily_aqi, lags=60, ax=axes[0], color='#2c3e50', alpha=0.05)
axes[0].set_title('ACF - Daily AQI (60 day lags)')
axes[0].axvline(7, color='red', ls='--', alpha=0.5, label='7-day')
axes[0].axvline(30, color='orange', ls='--', alpha=0.5, label='30-day')
axes[0].legend()

plot_pacf(daily_aqi, lags=60, ax=axes[1], color='#8e44ad', alpha=0.05, method='ywm')
axes[1].set_title('PACF - Daily AQI (60 day lags)')

plt.tight_layout()
plt.show()

print('Interpretation:')
print('- Slow ACF decay indicates non-stationarity (confirms ADF test)')
print('- PACF cuts off after lag 1-2, suggesting AR(1) or AR(2) component')
print('- Periodic bumps at lag 7 and 30 indicate weekly/monthly cycles')

## 10. Hypothesis Testing

In [ ]:
# Mann-Whitney U: Weekend vs Weekday
weekend_aqi = df[df['day_of_week'] >= 5]['aqi_value']
weekday_aqi = df[df['day_of_week'] < 5]['aqi_value']
stat, p = stats.mannwhitneyu(weekend_aqi, weekday_aqi)
print(f'Weekend vs Weekday (Mann-Whitney U):')
print(f'  Weekday median: {weekday_aqi.median():.1f}, Weekend median: {weekend_aqi.median():.1f}')
print(f'  U-statistic: {stat:.0f}, p-value: {p:.6f}')
print(f'  Conclusion: {"Significant" if p < 0.05 else "Not significant"} difference\n')

# Kruskal-Wallis: Seasonal comparison
season_groups = [df[df['season']==s]['aqi_value'] for s in ['Winter','Spring','Summer','Autumn']]
stat, p = stats.kruskal(*season_groups)
print(f'Seasonal Comparison (Kruskal-Wallis H):')
print(f'  Winter: {season_groups[0].median():.1f}, Spring: {season_groups[1].median():.1f}')
print(f'  Summer: {season_groups[2].median():.1f}, Autumn: {season_groups[3].median():.1f}')
print(f'  H-statistic: {stat:.1f}, p-value: {p:.2e}')
print(f'  Conclusion: {"Significant" if p < 0.05 else "Not significant"} seasonal differences\n')

# Spearman: Temperature vs AQI
rho, p = stats.spearmanr(df['temperature_c'], df['aqi_value'])
print(f'Temperature vs AQI (Spearman):')
print(f'  rho = {rho:.4f}, p-value = {p:.2e}')
print(f'  Interpretation: {"Negative" if rho < 0 else "Positive"} monotonic relationship')

## 11. Feature Importance (Mutual Information)

In [ ]:
from sklearn.feature_selection import mutual_info_regression

feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                if c not in ['aqi_value', 'year']]
X = df[feature_cols].fillna(0)
y = df['aqi_value']

mi_scores = mutual_info_regression(X, y, random_state=42, n_neighbors=5)
mi_df = pd.DataFrame({'feature': feature_cols, 'MI': mi_scores})
mi_df = mi_df.sort_values('MI', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 8))
top20 = mi_df.head(20)
ax.barh(range(len(top20)), top20['MI'].values, color='#3498db', alpha=0.8)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20['feature'].values)
ax.set_xlabel('Mutual Information')
ax.set_title('Top 20 Features by Mutual Information with AQI', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('\nTop 10 Features:')
print(mi_df.head(10).to_string(index=False))

## 12. Outlier Detection

In [ ]:
key_cols_outlier = ['aqi_value', 'pm25', 'pm10', 'no2', 'so2', 'co', 'o3',
                    'temperature_c', 'humidity_pct', 'wind_speed_ms']

print('IQR-Based Outlier Analysis:')
print(f'{"Variable":<18} {"Outliers":<10} {"Pct":<8} {"Lower":<10} {"Upper":<10}')
print('-' * 56)

for col in key_cols_outlier:
    data = df[col].dropna()
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    outliers = data[(data < lower) | (data > upper)]
    print(f'{col:<18} {len(outliers):<10} {len(outliers)/len(data)*100:<8.2f} {lower:<10.1f} {upper:<10.1f}')

# Hazardous events
high_aqi = df[df['aqi_value'] > 150]
print(f'\nHazardous Events (AQI > 150):')
print(f'  Count: {len(high_aqi):,} hours ({len(high_aqi)/len(df)*100:.1f}%)')
print(f'  Avg Temperature: {high_aqi["temperature_c"].mean():.1f} C')
print(f'  Avg Wind Speed: {high_aqi["wind_speed_ms"].mean():.1f} m/s')
print(f'  Avg Humidity: {high_aqi["humidity_pct"].mean():.1f}%')

## 13. Key Findings & Modeling Implications

### Summary of Key Findings

| Finding | Implication for Modeling |
|---------|------------------------|
| Non-Gaussian pollutant distributions | Use ensemble/tree-based models over linear assumptions |
| Strong multicollinearity (r > 0.95) | Regularization (Ridge/ElasticNet) or tree-based feature selection |
| Non-stationary time series (ADF p=0.24) | Include lag features and rolling statistics |
| Clear 24h periodicity in ACF | Cyclical temporal encoding (sin/cos) |
| Winter peaks (thermal inversions) | Season as important feature |
| Low wind speed during hazardous events | Wind-pollutant interaction features |
| Slow ACF decay + PACF cutoff at lag 2 | AR(2) component in the data generating process |
| 36.7% hazardous hours | Class imbalance consideration for alert systems |

In [ ]:
print('EDA Complete!')
print(f'\nDataset: {len(df):,} observations, {df.shape[1]} features')
print(f'Time span: 5 years (2021-2026)')
print(f'AQI range: {df["aqi_value"].min():.0f} - {df["aqi_value"].max():.0f}')
print(f'\nNext steps:')
print('  1. Feature selection based on MI scores')
print('  2. Train/test split with temporal ordering')
print('  3. Model training with cross-validation')
print('  4. SHAP analysis on best model')